In [6]:
import pandas as pd
from tdc.multi_pred import DTI
import os

# Ensure the splits folder exists in your data directory
os.makedirs('../data/splits', exist_ok=True)

def process_dataset(dataset_name):
    print(f"========================================")
    print(f"LOADING {dataset_name} DATASET")
    print(f"========================================")
    
    # Load Data
    data = DTI(name=dataset_name)
    df = data.get_data()
    
    # Deliverable 1: Basic Stats
    num_drugs = df['Drug'].nunique()
    num_targets = df['Target'].nunique()
    num_pairs = len(df)
    min_affinity = df['Y'].min()
    max_affinity = df['Y'].max()
    
    print(f"Total Pairs: {num_pairs}")
    print(f"Unique Drugs: {num_drugs}")
    print(f"Unique Proteins: {num_targets}")
    print(f"Affinity Range: {min_affinity:.4f} to {max_affinity:.4f}\n")
    
    # Deliverable 2: Build the four splits
    print(f"Generating splits for {dataset_name} (This may take a moment)...")
    splits = {
        'random': data.get_split(),
        'cold_drug': data.get_split(method='cold_split', column_name='Drug'),
        'cold_target': data.get_split(method='cold_split', column_name='Target'),
        'cold_pair': data.get_split(method='cold_split', column_name=['Drug', 'Target'])
    }
    
    # Save all splits to the data/splits folder
    for split_name, split_dict in splits.items():
        for fold in ['train', 'valid', 'test']:
            file_path = f"../data/splits/{dataset_name.lower()}_{split_name}_{fold}.csv"
            split_dict[fold].to_csv(file_path, index=False)
            
    print(f"All 12 CSV files saved for {dataset_name}.\n")
    
    # Deliverable 2: Sanity Checks (Crucial Step)
    print(f"Running Sanity Checks for {dataset_name}...")
    for split_name in ['cold_drug', 'cold_target', 'cold_pair']:
        train_df = splits[split_name]['train']
        test_df = splits[split_name]['test']
        
        if 'drug' in split_name or 'pair' in split_name:
            overlap = set(train_df['Drug']).intersection(set(test_df['Drug']))
            print(f"[{split_name}] Drug overlap between train/test: {len(overlap)}")
            if len(overlap) > 0: print(">>> WARNING: DRUG LEAKAGE DETECTED <<<")
                
        if 'target' in split_name or 'pair' in split_name:
            overlap = set(train_df['Target']).intersection(set(test_df['Target']))
            print(f"[{split_name}] Target overlap between train/test: {len(overlap)}")
            if len(overlap) > 0: print(">>> WARNING: TARGET LEAKAGE DETECTED <<<")
    print("\n")

# Run the process for both datasets
process_dataset('DAVIS')
process_dataset('KIBA')

Downloading...


LOADING DAVIS DATASET


100%|██████████| 21.4M/21.4M [00:49<00:00, 429kiB/s] 
Loading...
Done!


Total Pairs: 25772
Unique Drugs: 68
Unique Proteins: 379
Affinity Range: 0.0160 to 10000.0000

Generating splits for DAVIS (This may take a moment)...


Downloading...


All 12 CSV files saved for DAVIS.

Running Sanity Checks for DAVIS...
[cold_drug] Drug overlap between train/test: 0
[cold_target] Target overlap between train/test: 0
[cold_pair] Drug overlap between train/test: 0
[cold_pair] Target overlap between train/test: 0


LOADING KIBA DATASET


100%|██████████| 96.6M/96.6M [03:20<00:00, 480kiB/s] 
Loading...
Done!


Total Pairs: 117657
Unique Drugs: 2068
Unique Proteins: 229
Affinity Range: 0.0000 to 17.2002

Generating splits for KIBA (This may take a moment)...
All 12 CSV files saved for KIBA.

Running Sanity Checks for KIBA...
[cold_drug] Drug overlap between train/test: 0
[cold_target] Target overlap between train/test: 0
[cold_pair] Drug overlap between train/test: 0
[cold_pair] Target overlap between train/test: 0




In [ ]:
import pandas as pd
import re
import os

file_path = '../data/raw/BindingDB_All.tsv'
output_path = '../data/raw/bindingdb_antiviral.csv'

# The exact keywords requested by your guide
keywords = [
    r"sars-cov-2 main protease", r"mpro", r"3c-like proteinase", 
    r"sars-cov-2 rna-dependent rna polymerase", r"rdrp", 
    r"hiv-1 protease", r"hiv-1 reverse transcriptase", 
    r"influenza neuraminidase"
]
pattern = '|'.join(keywords)

print(f"Reading BindingDB in chunks to protect RAM. This will take a few minutes...")

# Read the massive file in chunks of 50,000 rows
chunk_iterator = pd.read_csv(file_path, sep='\t', chunksize=50000, on_bad_lines='skip', low_memory=False, dtype=str)

first_chunk = True
total_found = 0

for i, chunk in enumerate(chunk_iterator):
    # Find the target name columns dynamically
    target_cols = [c for c in chunk.columns if 'Target Name' in c]
    if not target_cols:
        target_cols = chunk.columns # Fallback to search all columns
        
    mask = pd.Series(False, index=chunk.index)
    for col in target_cols:
        mask = mask | chunk[col].fillna('').str.contains(pattern, flags=re.IGNORECASE, regex=True)
        
    filtered = chunk[mask]
    total_found += len(filtered)
    
    if len(filtered) > 0:
        filtered.to_csv(output_path, mode='a', index=False, header=first_chunk)
        first_chunk = False
        
    if i % 10 == 0 and i > 0:
        print(f"Processed {i * 50000} rows... found {total_found} matches so far.")

print(f"\nExtraction complete! Found a total of {total_found} antiviral pairs.")
print(f"Saved deliverable to: {output_path}")